In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error

# 👇 これを1番最初のセルで実行して「ストック」しておく
def run_regression_audition(X, y, test_size=0.2, random_state=42):
    """
    生のヒント(X)と答え(y)を渡すだけで、
    前処理(標準化)から3つのモデルの一括評価までを自動で行う関数
    """
    # 1. データの分割
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=random_state)
    
    # 2. パイプラインの定義（完全固定の定型文）
    pipelines = {
        'LinearRegression（線形回帰）': Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())]),
        'RandomForest（ランダムフォレスト）': Pipeline([('scaler', StandardScaler()), ('rf', RandomForestRegressor(random_state=random_state))]),
        'SVR（サポートベクターマシン）': Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
    }
    
    print("--- パイプライン版・一括計測 ---")
    results = {}
    for name, pipe in pipelines.items():
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_val)
        error = root_mean_squared_error(y_val, y_pred)
        print(f"{name} の平均誤差: {error:.4f}")
        results[name] = pipe # 後から使えるように学習済みのモデルも保存しておく
        
    return results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. オーディション結果から、一番優秀だったランダムフォレストのパイプラインを取り出す
best_pipe = trained_models['RandomForest（ランダムフォレスト）']

# 2. パイプラインの中から、AI本体（rf）を取り出す
rf_model = best_pipe.named_steps['rf']

# 3. AIが計算した「各特徴量の重要度」をゲットする
importances = rf_model.feature_importances_
feature_names = X_train.columns

# 4. 見やすいようにデータフレーム（表）にまとめる
df_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False) # 重要度が高い順に並び替え

# 5. グラフの描画（日本語が文字化けしないようにシンプルに設定）
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=df_importance, palette='viridis')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance (Score)')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

# 数値でも確認できるように上位を表示
print(df_importance)

In [ ]:
# 1. 箱から「線形回帰」のパイプラインを取り出す（★ここを変更）
best_pipe = trained_models['LinearRegression（線形回帰）']

# 2. パイプラインの中から、AI本体（lr）を取り出す（★ここを lr に変更）
lr_model = best_pipe.named_steps['lr']

# 3. 🌟ここが仕組みの違い！線形回帰では「coef_（傾き・係数）」を使います
importances = lr_model.coef_  # ★ feature_importances_ ではなく coef_ になります
feature_names = X_train.columns

# 4. 見やすいようにデータフレームにまとめる
df_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 5. グラフの描画
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=df_importance, palette='coolwarm') # 色味を少し変えました
plt.title('Feature Importance (Linear Regression - Coefficients)')
plt.xlabel('Coefficient Value (影響度)')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

print(df_importance)